In [1]:
import sys
import os

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import datasets
from functools import partial

import torch

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
model_path = "../../self-corrective-llama_untrained"
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

model_config = AutoConfig.from_pretrained(model_path)
model_config.alpha_boost = 5.0
model_config.tau = 0.7
model_config.max_boost = 8.0

model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True,)

print(tokenizer.encode("<DEL_S>"))
print(tokenizer.encode("<DEL_A>"))

[128000, 128256]
[128000, 128257]


In [3]:
dataset = datasets.load_from_disk("../../dataset/training")
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 31519
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 3503
})


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
sample = train_dataset[0]
sample["input_ids"][391] = 128256
sample["input_ids"][392] = 128257

sample["labels"][391] = 128256
sample["labels"][392] = 128257

sample["hallucination_labels"][391] = 1
sample["hallucination_labels"][392] = 2


sample["input_ids"] = torch.tensor([sample["input_ids"][390:420]])
sample["attention_mask"] = torch.tensor([sample["attention_mask"][390:420]])
sample["labels"] = torch.tensor([sample["labels"][390:420]])
sample["hallucination_labels"] = torch.tensor([sample["hallucination_labels"][390:420]])
print(sample)

{'input_ids': tensor([[   315, 128256, 128257,    393,   7616,     82,    315,   5684,  25485,
            340,     28,    220,    508,    482,    320,    605,    489,    220,
             20,    489,    220,     19,    340,     28,    220,    508,    482,
            220,    777,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]]), 'labels': tensor([[   315, 128256, 128257,    393,   7616,     82,    315,   5684,  25485,
            340,     28,    220,    508,    482,    320,    605,    489,    220,
             20,    489,    220,     19,    340,     28,    220,    508,    482,
            220,    777,    198]]), 'hallucination_labels': tensor([[0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0]])}


In [17]:
model.forward(
    input_ids=sample["input_ids"], 
    attention_mask=sample["attention_mask"], 
    labels=sample["labels"], 
    hallucination_labels=sample["hallucination_labels"]
)

clamped_input_ids: tensor([[   315, 128255, 128255,    393,   7616,     82,    315,   5684,  25485,
            340,     28,    220,    508,    482,    320,    605,    489,    220,
             20,    489,    220,     19,    340,     28,    220,    508,    482,
            220,    777,    198]])
special_token_mask: tensor([[False,  True,  True, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False]])
special_ids: tensor([0, 1])
main_logits: tensor([[[-2.4917, -2.4910, -2.4910, -2.4922, -2.4919, -2.4920],
         [-0.8888, -0.8886, -0.8880, -0.8890, -0.8893, -0.8893],
         [-2.9848, -2.9845, -2.9846, -2.9849, -2.9847, -2.9852],
         [-0.1085, -0.1090, -0.1085, -0.1094, -0.1083, -0.1084],
         [-1.1514, -1.1515, -1.1510, -1.1517, -1.1509, -1.1509],
         [-2.3767, -2.3767, -2.3764, -2.3771, -2.3766, -2.3768],
         [-3.

SelfCorrectiveLlamaOutput(loss=None, logits=tensor([[[ 4.3350e+00,  4.9669e+00,  6.1672e+00,  ..., -2.4920e+00,
           3.7504e-03,  3.7504e-03],
         [ 8.5463e+00,  6.9778e+00,  6.1007e+00,  ..., -8.8927e-01,
           1.0100e+00, -4.2067e-03],
         [-1.1259e+00, -4.2584e-02,  8.5481e-01,  ..., -2.9852e+00,
          -2.2446e-03,  1.7024e+00],
         ...,
         [ 5.2250e+00,  5.6446e+00,  7.1412e+00,  ..., -6.1765e-01,
           3.6248e-03,  3.6248e-03],
         [ 8.3539e+00,  1.0444e+01,  9.9818e+00,  ..., -3.0058e-01,
           2.1220e-03,  2.1220e-03],
         [ 6.8168e+00,  1.0098e+01,  1.2364e+01,  ..., -2.5683e-01,
          -2.1794e-03, -2.1794e-03]]], grad_fn=<CopySlices>), past_key_values=DynamicCache(layers=[<transformers.cache_utils.DynamicLayer object at 0x32d51e710>, <transformers.cache_utils.DynamicLayer object at 0x107f676d0>, <transformers.cache_utils.DynamicLayer object at 0x32d78dfd0>, <transformers.cache_utils.DynamicLayer object at 0x32d78d690>

In [23]:
text = "What is the capital of France?"
sample = train_dataset[0]
input_ids = tokenizer.encode(text)
# input_ids = torch.tensor([input_ids])
# attention_mask = torch.ones_like(input_ids)

input_ids = torch.tensor([sample["input_ids"]])
attention_mask = torch.ones_like(torch.tensor([sample["attention_mask"]]))

result = model.generate(input_ids=input_ids, attention_mask=attention_mask)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


clamped_input_ids: tensor([[128000, 128006,   9125, 128007,    271,   2675,    527,    264,  96278,
          15592,  21651,   1122,     13,   4718,   3465,    374,    311,  11886,
            279,   2768,   7033,   3575,    382,  12763,   1521,   7504,  15884,
            512,     16,     13,   3146,   2127,  56956,    279,   3575,  68063,
           5629,     11,   3619,    279,   2728,   2038,    323,   1148,    374,
           1694,   4691,    627,     17,     13,   3146,   5733,    434,   2092,
             85,   2968,  68063,  31001,    422,    279,   3575,    374,   2092,
          24694,     13,    362,   3575,   2643,    387,   7120,  89197,    422,
            433,    596,   3900,  31356,     11,   5727,  81523,     11,    477,
          37856,   5995,   2038,    627,     18,     13,   3146,     50,   4035,
            477,  83017,     25,   1035,    256,    482,   3146,   2746,   2092,
          24694,  68063,  40665,    264,   3094,  14656,  30308,   6425,     11,
         

In [25]:
res = tokenizer.decode(result[0])
print(res)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a meticulous AI mathematician. Your task is to solve the following math problem.

Follow these steps carefully:
1. **Analyze the problem:** First, understand the given information and what is being asked.
2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.
3. **Solve or Explain:**
   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.
   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.

Your entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.
If you realize you have made a mistake, you must use one of the following tools to correct it: Use <DE